# 02 · Label generation (PRD §7)
**Runs:** locally. **Needs:** `GEMINI_API_KEY` (and `GEMINI_MODEL`) in the repo `.env`.
**Produces:** `data/labels/run_primary.json` (resumable checkpoint).

The prompt is the PRD §7.2 prompt verbatim; the star rating is never sent. Re-running
this notebook resumes from the checkpoint — completed labels are never re-requested.
`MAX_REQUESTS` is a hard budget including retries.

In [ ]:
# --- Repo setup: make `app` (backend) and `vera_training` importable ---------
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "PRD.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
for sub in ("backend", "training"):
    if str(ROOT / sub) not in sys.path:
        sys.path.insert(0, str(ROOT / sub))

import os
if not (ROOT / ".env").exists():  # app settings require these; real values come from .env
    os.environ.setdefault("DATABASE_URL", "sqlite:///:memory:")
    os.environ.setdefault("REDIS_URL", "redis://localhost:6379/0")

from vera_training import paths
paths.ensure_dirs()
print("Repo root:", ROOT)

In [ ]:
from dotenv import load_dotenv
import pandas as pd
from vera_training.labeling import Provider, run_labels

load_dotenv(ROOT / ".env")
pool = pd.read_parquet(paths.LABELING_POOL)

MODEL = os.environ.get("GEMINI_MODEL", "gemini-2.5-flash")
MAX_REQUESTS = 2600          # 2000 labels + retry headroom
REQUESTS_PER_MINUTE = 10     # stay well inside free-tier rate limits

provider = Provider("gemini", MODEL)
state = run_labels(
    pool, paths.PRIMARY_RUN, provider,
    provider_name="gemini", model=MODEL, run_id="primary", seed=1,
    max_requests=MAX_REQUESTS, requests_per_minute=REQUESTS_PER_MINUTE,
)
print(f"labels: {len(state['labels'])}/{len(pool)}  requests used: {state['requests']}")
print("rows with errors:", len(state["errors"]))

## Label distribution (should be far less skewed than the stars)

In [ ]:
scores = pd.Series({k: v["experience_score"] for k, v in state["labels"].items()})
stars = pool.set_index("review_id").rating.reindex(scores.index)
print(scores.value_counts().sort_index())
print("mean |star - label|:", (stars - scores).abs().mean().round(3))
print("irony/sarcasm flagged:", sum(v["has_irony_or_sarcasm"] for v in state["labels"].values()))